# Shane Waldron
### Kaggle Playground Series Homework
### GSB 545

#### Notebooks to reference
1. https://www.kaggle.com/code/logeshwari12345/predicting-irrigation-need

This notebook provides good ideas for data preprocessing in terms of making use of Column Transformers and Pipelines that we used in GSB 544.

2. https://www.kaggle.com/code/aymankhan555/irrigation-prediction-eda-xgb

This notebook has interesting EDA insights, like checking the distribution of the target variable and histograms and box and whisker plots of somee of the numerical predictors.

3. https://www.kaggle.com/code/anilabukhari/irrigation-need-eda-feature-analysis-insights

More interesting EDA insights and strategies, I tend to not focus enough on EDA so notebooks 2 and 3 will be helpful for me.

### Exploratory Data Analysis Notebook
Link: https://github.com/swaldron12/gsb545/blob/main/Kaggle%20Competition/EDA.ipynb

### Modeling Notebook

Link: https://github.com/swaldron12/gsb545/blob/main/Kaggle%20Competition/Modeling.ipynb


I tried two tree-based approaches: a tuned RandomForestClassifier and a tuned XGBClassifier, both inside a preprocessing pipeline that one-hot encoded the categorical variables and passed the numeric variables through since tree based methods don't requring scaling of numerical variables. For XGBoost, I also used balanced sample weights to help with the minority class.

Both models worked well overall and produced very similar top-line results: each reached 0.98 test accuracy and 0.98 weighted F1. The random forest was slightly more conservative, with stronger precision on class 2 (High) at 0.98, but lower recall at 0.88, so it missed more truly high-irrigation cases. XGBoost improved class 2 recall to 0.94, which was the most meaningful improvement in the notebook, although its precision on that class dropped to 0.91. So the improvement was small overall, but meaningful if catching high-irrigation cases mattered most.


| Model         | Accuracy | Weighted F1 | Class 2 Precision | Class 2 Recall | Takeaway                                      | Leaderboard Score |
|---------------|----------|-------------|-------------------|----------------|-----------------------------------------------|-------------------|
| Random Forest | 0.98     | 0.98        | 0.98              | 0.88           | Very strong overall, but missed more High cases | 0.96748           |
| XGBoost       | 0.98     | 0.98        | 0.91              | 0.94           | Similar overall, better at finding High cases  | 0.95149           |


In terms of boosting vs. bagging, boosting did not create a big overall perforamnce jump but it gave a meaningful advantage when detecting high irrigation need which was the hardest class to predict.

### Phase 2 Plan

Due to lack of time and resources, I am going to try using Optuna to tune my next set of models for my submission. I tried using it but the training was taking too long (likely up to 5 hours for Random Forest). If I set aside more time I can likely make that work, as I did see that the intitial scores from the Optuna training were quite high. I also will try out a few more models, maybe some that train quicker so I can test out Optuna fully.

## Boosting Models
Link: https://github.com/swaldron12/gsb545/blob/main/Kaggle%20Competition/Boosting-Models.ipynb


| Model               | Accuracy | Weighted F1 | Class 2 Precision | Class 2 Recall | Takeaway                                                  | Leaderboard Score|
|---------------------|----------|-------------|-------------------|----------------|-----------------------------------------------------------|------------------|
| XGBoost             | 0.98     | 0.98        | 0.97              | 0.92           | Very strong overall; high irrigation need remained hardest | 0.95892         |
| LightGBM            | 0.98     | 0.98        | 0.97              | 0.92           | Performed essentially identically to XGBoost              | 0.95871          |
| LightGBM (Strat CV) | 0.98     | 0.98        | 0.97              | 0.92           | No meaningful improvement from stratified CV tuning       | Not submitted    |

I tried two boosting models, XGBoost and LightGBM, both with one-hot encoding for categorical variables and Optuna for hyperparameter tuning. Both worked very well: each reached 0.98 accuracy and 0.98 weighted F1 on the holdout set, and both predicted low and medium irrigation need especially well. The hardest class was high irrigation need, where both models had strong precision (0.97) but lower recall (0.92), so some high-need cases were still missed.

Overall, the improvements between models were small. XGBoost had the best cross-validation score (0.98464) compared with LightGBM (0.98457 and 0.98433 for the two tuning setups), but these differences were minimal. In practice, the models performed almost identically, so boosting worked well, but no single model showed a meaningful advantage.

## Feature Engineering
I tried feature engineering based on my EDA, including dryness_score, stress_count, weather thresholds, moisture/rainfall flags, and interaction terms. These worked well overall, especially stress_count, active_growth_stage, high_temperature, high_wind, and low_rainfall, which appeared highly important in XGBoost.

I trained XGBoost and LightGBM models using the full engineered feature set, then used feature importance from both models to create a selected-feature version. XGBoost improved slightly with selected features, while LightGBM performed slightly worse, so feature selection was useful but not clearly better overall.
| Model | Features | Holdout Accuracy | Kaggle Score |
|---|---|---:|---:|
| XGBoost | Full | 0.984794 | -- |
| LightGBM | Full | 0.985032 | -- |
| XGBoost | Selected | 0.984963 | -- |
| LightGBM | Selected | 0.984714 | -- |
| XGB + LGBM Ensemble | Selected | 0.985132 | -- |
| XGB + LGBM Ensemble | Full | 0.985201 | 0.95947 |


The best model was the full-feature probability averaging ensemble with accuracy 0.985201. The improvement over LightGBM alone was small, but still useful for a Kaggle competition. Moving forward, I would continue using the engineered stress features and probability averaging, while keeping the full feature set unless leaderboard results show selected features perform better.